In [17]:
import pandas as pd
import numpy as np

In [18]:
# AM import (both log and behavior)
behav_am_df = pd.read_excel(
    "C:/Users/HELIOS-300/Desktop/Data/am_behposture_onesheet.xlsx",
    engine="openpyxl"
)
log_df = pd.read_csv(
    "C:/Users/HELIOS-300/Desktop/Data/DO_LOG_final_UPDATED.csv",
    encoding="utf-8"
)

# Keep only State start rows (as you’re doing now)
behav_am_df = behav_am_df[behav_am_df["Event_Type"] == "State start"].copy()

# AM24/AM26 DO2 duplicate-observation cleanup:
# - Keep AM24DO2_R_FINAL_C and drop AM24DO2_M_FINAL_R.
# - Drop ALL AM26DO2_R_copyB_FINAL_C rows (regardless of negative values).
if "Observation" in behav_am_df.columns:
    _obs_norm = behav_am_df["Observation"].astype("string").str.strip().str.upper()
    _drop_am24_do2_m = _obs_norm.eq("AM24DO2_M_FINAL_R")
    _drop_am26_do2_b = _obs_norm.eq("AM26DO2_R_COPYB_FINAL_C")

    print("Rows dropped for AM24DO2_M_FINAL_R:", int(_drop_am24_do2_m.sum()))
    print("Rows dropped for AM26DO2_R_copyB_FINAL_C:", int(_drop_am26_do2_b.sum()))

    behav_am_df = behav_am_df.loc[~(_drop_am24_do2_m | _drop_am26_do2_b)].copy()

# Targeted AM11 DO1 copyB negative-time cleanup (must run BEFORE global negative drop):
# For AM11DO1_N_copyB_FINAL_C rows with negative Time_Relative_hms,
# keep only the LES-screen row at hour 13, and reset it to relative start.
if all(c in behav_am_df.columns for c in ["Observation", "Behavior", "Time_Absolute_hms", "Time_Relative_hms"]):
    _obs = behav_am_df["Observation"].astype("string").str.strip().str.upper()
    _beh = behav_am_df["Behavior"].astype("string").str.strip()
    _trh_td = pd.to_timedelta(
        behav_am_df["Time_Relative_hms"].astype(str).str.strip(),
        errors="coerce"
    )
    _is_target_obs = _obs.eq("AM11DO1_N_COPYB_FINAL_C")
    _is_neg = _trh_td < pd.Timedelta(0)

    _tah = behav_am_df["Time_Absolute_hms"].astype("string").str.strip()
    _hour13 = _tah.str.startswith("13").fillna(False)
    _is_target_behavior = _beh.eq("LES- screen based leisure time (TV, video game, computer)")

    _keep_special = _is_target_obs & _is_neg & _hour13 & _is_target_behavior
    _drop_special = _is_target_obs & _is_neg & ~_keep_special

    print("AM11 DO1 copyB negative rows dropped (targeted):", int(_drop_special.sum()))
    print("AM11 DO1 copyB negative rows kept/reset to zero:", int(_keep_special.sum()))

    behav_am_df = behav_am_df.loc[~_drop_special].copy()

    # Reset surviving special row(s) to session start
    behav_am_df.loc[_keep_special, "Time_Relative_hms"] = "00:00:00"
    if "Time_Relative_hmsf" in behav_am_df.columns:
        behav_am_df.loc[_keep_special, "Time_Relative_hmsf"] = "00:00:00"
    if "Time_Relative_sf" in behav_am_df.columns:
        behav_am_df.loc[_keep_special, "Time_Relative_sf"] = 0

# Drop rows where Time_Relative_hms is negative (requested experiment).
# We parse as timedelta first so string variants are handled consistently.
if "Time_Relative_hms" in behav_am_df.columns:
    _trh = pd.to_timedelta(
        behav_am_df["Time_Relative_hms"].astype(str).str.strip(),
        errors="coerce"
    )
    _neg_mask = _trh < pd.Timedelta(0)
    print("Rows dropped for negative Time_Relative_hms:", int(_neg_mask.sum()))
    behav_am_df = behav_am_df.loc[~_neg_mask].copy()

# Convert timedelta columns back to time strings (HH:MM:SS format)
# Fixes the "0 days 00:00:00" display issue
for col in behav_am_df.columns:
    if pd.api.types.is_timedelta64_dtype(behav_am_df[col]):
        base_date = pd.Timestamp("1900-01-01")
        behav_am_df[col] = (base_date + behav_am_df[col]).dt.strftime("%H:%M:%S")

# ============================================================
# log_df: special AM setup / cleanup
# ============================================================

# Extract numeric id from log_df['id'] values like "AM02"
log_df["id"] = log_df["id"].astype(str).str.extract(r"AM(\d{2})", expand=False)
log_df["id"] = pd.to_numeric(log_df["id"], errors="coerce").astype("int64")

# Use obs (already formatted like DO2_a, DO2_b, DO1_a, etc.) as authoritative do label
log_df["do"] = log_df["obs"].astype(str).str.strip()

# AM10 DO2 manual log override
# Requested window: 11:20:00 -> 13:21:00 (length 2:01:00)
_m_am10_do2 = log_df["id"].eq(10) & log_df["do"].eq("DO2")
if _m_am10_do2.any():
    log_df.loc[_m_am10_do2, "start_time"] = "11:20:00"
    log_df.loc[_m_am10_do2, "stop_time"] = "13:21:00"
    if "duration" in log_df.columns:
        log_df.loc[_m_am10_do2, "duration"] = "2:01:00"
    print("Applied AM10 DO2 log override rows:", int(_m_am10_do2.sum()))

# Fix duplicated DO2 segments for id=11 and id=26 ONLY if they still appear as plain "DO2"
# (In your latest log, these are already DO2_a / DO2_b, so this will do nothing.)
for _id in [11, 26]:
    m = log_df["id"].eq(_id) & log_df["do"].eq("DO2")
    if m.sum() >= 2:
        tmp = log_df.loc[m].copy()
        tmp["_start_td"] = pd.to_timedelta(tmp["start_time"].astype(str).str.strip(), errors="coerce")
        idx_sorted = tmp.sort_values("_start_td", kind="mergesort").index

        if len(idx_sorted) >= 1:
            log_df.loc[idx_sorted[0], "do"] = "DO2_a"
        if len(idx_sorted) >= 2:
            log_df.loc[idx_sorted[1], "do"] = "DO2_b"
        if len(idx_sorted) > 2:
            for j in range(2, len(idx_sorted)):
                log_df.loc[idx_sorted[j], "do"] = f"DO2_b{j-1}"

# ============================================================
# behav_am_df: special AM setup / cleanup
# ============================================================

# id = 2 digits after AM in Observation
behav_am_df["id"] = (
    behav_am_df["Observation"]
    .str.extract(r"AM(\d{2})", expand=False)
    .astype("int64")
)

# do label (DO1, DO1_a, DO1_b, DO2, DO2_a, DO2_b, etc.)
behav_am_df["do"] = (
    behav_am_df["Observation"]
    .str.extract(r"(DO\d+(?:_[ab])?)", expand=False)
    .astype("string")
    .str.strip()
)

# normalize suffix case if it ever appears
behav_am_df["do"] = (
    behav_am_df["do"]
    .str.replace("_A", "_a", regex=False)
    .str.replace("_B", "_b", regex=False)
)

# For behav rows labeled DO2 but log has DO2_a / DO2_b (id=11, id=26),
# split using absolute timestamps (only affects rows where do == "DO2")
behav_am_df["_abs_dt_behav"] = pd.to_datetime(
    behav_am_df["Date_Time_Absolute_dmy_hmsf"],
    errors="coerce"
)

splits = {
    11: pd.Timestamp("2017-09-05 11:03:00"),
    26: pd.Timestamp("2018-02-24 17:13:00"),
}

for _id, cut in splits.items():
    m = (
        behav_am_df["id"].eq(_id)
        & behav_am_df["do"].eq("DO2")
        & behav_am_df["_abs_dt_behav"].notna()
    )
    behav_am_df.loc[m & (behav_am_df["_abs_dt_behav"] < cut), "do"] = "DO2_a"
    behav_am_df.loc[m & (behav_am_df["_abs_dt_behav"] >= cut), "do"] = "DO2_b"

# ============================================================
# NEW: add do_base for both dfs (strip only trailing _a/_b)
# This is what we’ll use for joins + the second-by-second backbone.
# ============================================================

log_df["do_base"] = (
    log_df["do"].astype(str).str.strip()
    .str.replace(r"_(a|b)$", "", regex=True)
)

behav_am_df["do_base"] = (
    behav_am_df["do"].astype(str).str.strip()
    .str.replace(r"_(a|b)$", "", regex=True)
)

# (keep _abs_dt_behav for QC; drop later if you want)
# behav_am_df.drop(columns=["_abs_dt_behav"], inplace=True, errors="ignore")

Rows dropped for AM24DO2_M_FINAL_R: 181
Rows dropped for AM26DO2_R_copyB_FINAL_C: 153
AM11 DO1 copyB negative rows dropped (targeted): 6
AM11 DO1 copyB negative rows kept/reset to zero: 0
Rows dropped for negative Time_Relative_hms: 89
Applied AM10 DO2 log override rows: 1


In [19]:
import pandas as pd
import numpy as np
import itertools

# ============================================================
# CHUNK 1: Rebuild absolute time from log_df anchors + relative offsets
#
# Strategy:
#   1) Map each Observation to a log row (id + do_base, duration-guided)
#   2) Normalize relative seconds per Observation to start at 0
#   3) Anchor to log start_dt to reconstruct absolute time columns
#   4) Build sec_by_sec from reconstructed absolute starts (inclusive end)
# ============================================================


def _union_coverage_seconds(g):
    g2 = g.dropna(subset=["rel_s", "rel_end_s"]).sort_values("rel_s")
    if g2.empty:
        return np.nan
    arr = g2[["rel_s", "rel_end_s"]].to_numpy(dtype=float)
    cur_s, cur_e = arr[0]
    total = 0.0
    for s, e in arr[1:]:
        if s <= cur_e:
            cur_e = max(cur_e, e)
        else:
            total += cur_e - cur_s
            cur_s, cur_e = s, e
    total += cur_e - cur_s
    return total


# ---------- Prepare log anchors ----------
log_work = log_df.copy()
if "id_num" not in log_work.columns:
    log_work["id_num"] = pd.to_numeric(log_work["id"], errors="coerce")
else:
    log_work["id_num"] = pd.to_numeric(log_work["id_num"], errors="coerce")

if "do_base" not in log_work.columns:
    log_work["do_base"] = (
        log_work["do"].astype(str).str.strip().str.replace(r"_(a|b)$", "", regex=True)
    )

log_work["start_date"] = pd.to_datetime(
    dict(year=log_work["start_year"], month=log_work["start_month"], day=log_work["start_day"]),
    errors="coerce",
)
log_work["start_dt"] = pd.to_datetime(
    log_work["start_date"].dt.strftime("%Y-%m-%d") + " " + log_work["start_time"].astype(str),
    errors="coerce",
)
log_work["stop_td"] = pd.to_timedelta(log_work["stop_time"].astype(str).str.strip(), errors="coerce")
log_work["stop_dt"] = log_work["start_date"] + log_work["stop_td"]
overnight = log_work["stop_dt"] < log_work["start_dt"]
log_work.loc[overnight, "stop_dt"] = log_work.loc[overnight, "stop_dt"] + pd.Timedelta(days=1)
log_work["dur_log_s"] = (log_work["stop_dt"] - log_work["start_dt"]).dt.total_seconds()


# ---------- Prepare behavior rows ----------
df = behav_am_df.copy()

# Relative offsets derived from absolute datetime (more stable than Time_Relative_*).
df["_abs_dt_raw"] = pd.to_datetime(df["Date_Time_Absolute_dmy_hmsf"], errors="coerce")

# Fallback parse from Date_dmy + Time_Absolute_hms if needed.
if "Date_dmy" in df.columns and "Time_Absolute_hms" in df.columns:
    miss_abs = df["_abs_dt_raw"].isna()
    if miss_abs.any():
        fallback_dt = pd.to_datetime(
            df.loc[miss_abs, "Date_dmy"].astype(str).str.strip()
            + " "
            + df.loc[miss_abs, "Time_Absolute_hms"].astype(str).str.strip(),
            errors="coerce",
            dayfirst=True,
        )
        df.loc[miss_abs, "_abs_dt_raw"] = fallback_dt

df["_abs_dt_sec"] = df["_abs_dt_raw"].dt.floor("s")
abs_start_per_obs = df.groupby("Observation")["_abs_dt_sec"].transform("min")
df["rel_s"] = (df["_abs_dt_sec"] - abs_start_per_obs).dt.total_seconds()

print("Rows with missing parsed absolute datetime:", int(df["_abs_dt_raw"].isna().sum()))
print("Rows with negative rel_s after abs-derivation:", int((df["rel_s"] < 0).sum()))

# Duration in seconds (ceil), missing -> 0
df["_dur_s"] = pd.to_numeric(df["Duration_sf"], errors="coerce").fillna(0.0)
df["_dur_s_int"] = np.ceil(df["_dur_s"]).astype("int64")
df["rel_end_s"] = df["rel_s"] + df["_dur_s_int"]

# Observation-level summary for mapping
obs_sum = (
    df.groupby("Observation", as_index=False)
    .agg(
        id_num=("id", "first"),
        do=("do", "first"),
        do_base=("do_base", "first"),
        rel_min=("rel_s", "min"),
        rel_max=("rel_end_s", "max"),
        n_rows=("Observation", "size"),
    )
    .copy()
)
obs_sum["union_cov_s"] = (
    df.groupby("Observation", sort=False)
    .apply(_union_coverage_seconds)
    .reindex(obs_sum["Observation"])
    .to_numpy()
)


# ---------- Map observations to log rows ----------
mapped_rows = []
for (pid, db), g_obs in obs_sum.groupby(["id_num", "do_base"], dropna=False):
    g_obs = g_obs.sort_values("Observation", kind="mergesort").reset_index(drop=True)
    g_log = log_work[(log_work["id_num"].eq(pid)) & (log_work["do_base"].eq(db))].copy()
    g_log = g_log.sort_values("start_dt", kind="mergesort").reset_index(drop=True)

    if g_log.empty:
        for _, ro in g_obs.iterrows():
            mapped_rows.append(
                {
                    **ro.to_dict(),
                    "map_status": "missing_log_group",
                    "log_obs": pd.NA,
                    "log_start_dt": pd.NaT,
                    "log_stop_dt": pd.NaT,
                    "dur_log_s": np.nan,
                }
            )
        continue

    vals_obs = g_obs["union_cov_s"].to_numpy(dtype=float)
    vals_log = g_log["dur_log_s"].to_numpy(dtype=float)
    n_obs = len(g_obs)
    n_log = len(g_log)
    pairs = []  # (obs_idx, log_idx)

    # One-to-one assignment only (no log-row reuse).
    # If there are more observations than log rows, extras are left unmapped.
    if n_obs <= 8 and n_log <= 8:
        if n_obs <= n_log:
            best_cost = np.inf
            best_perm = None
            for perm in itertools.permutations(range(n_log), n_obs):
                cost = np.abs(vals_obs - vals_log[list(perm)]).sum()
                if cost < best_cost:
                    best_cost = cost
                    best_perm = perm
            pairs = [(i, best_perm[i]) for i in range(n_obs)]
        else:
            best_cost = np.inf
            best_obs_subset = None
            best_perm = None
            for obs_subset in itertools.combinations(range(n_obs), n_log):
                obs_vals = vals_obs[list(obs_subset)]
                for perm in itertools.permutations(range(n_log), n_log):
                    cost = np.abs(obs_vals - vals_log[list(perm)]).sum()
                    if cost < best_cost:
                        best_cost = cost
                        best_obs_subset = obs_subset
                        best_perm = perm
            pairs = [(best_obs_subset[i], best_perm[i]) for i in range(n_log)]
    else:
        # Greedy fallback for larger groups
        if n_obs <= n_log:
            remaining_logs = set(range(n_log))
            for i in range(n_obs):
                j = min(remaining_logs, key=lambda k: abs(vals_obs[i] - vals_log[k]))
                remaining_logs.remove(j)
                pairs.append((i, j))
        else:
            remaining_obs = set(range(n_obs))
            for j in range(n_log):
                i = min(remaining_obs, key=lambda k: abs(vals_obs[k] - vals_log[j]))
                remaining_obs.remove(i)
                pairs.append((i, j))

    matched_obs = set()
    for i, j in pairs:
        matched_obs.add(i)
        ro = g_obs.iloc[i]
        rl = g_log.iloc[j]
        mapped_rows.append(
            {
                **ro.to_dict(),
                "map_status": "one_to_one_unique_assignment",
                "log_obs": rl["obs"],
                "log_start_dt": rl["start_dt"],
                "log_stop_dt": rl["stop_dt"],
                "dur_log_s": rl["dur_log_s"],
            }
        )

    # Extra observations beyond available log rows are explicitly unmapped.
    for i in range(n_obs):
        if i in matched_obs:
            continue
        ro = g_obs.iloc[i]
        mapped_rows.append(
            {
                **ro.to_dict(),
                "map_status": "unmapped_extra_observation",
                "log_obs": pd.NA,
                "log_start_dt": pd.NaT,
                "log_stop_dt": pd.NaT,
                "dur_log_s": np.nan,
            }
        )

map_df = pd.DataFrame(mapped_rows)

# Duration QC at mapping level
map_df["dur_diff_s"] = map_df["union_cov_s"] - map_df["dur_log_s"]
map_df["dur_abs_diff_s"] = map_df["dur_diff_s"].abs()

print("Mapping status:")
print(map_df["map_status"].value_counts(dropna=False).to_string())

ok_map = map_df[map_df["dur_log_s"].notna()].copy()
if not ok_map.empty:
    print("\nDuration abs diff summary (union coverage vs log):")
    print(ok_map["dur_abs_diff_s"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_string())

miss_map = map_df[map_df["map_status"].eq("missing_log_group")]
print("\nObservations missing log mapping:", len(miss_map))
if len(miss_map):
    print(miss_map[["Observation", "id_num", "do", "do_base"]].to_string(index=False))


# ---------- Rebuild absolute columns from log start + normalized relative ----------
df = df.merge(
    map_df[["Observation", "log_start_dt", "log_stop_dt", "dur_log_s", "log_obs", "map_status", "rel_min"]],
    on="Observation",
    how="left",
    validate="many_to_one",
)

# Normalize per Observation so each starts at 0 at the mapped log start
df["rel_norm_s"] = df["rel_s"] - df["rel_min"]
df["Date_Time_Absolute_dmy_hmsf"] = df["log_start_dt"] + pd.to_timedelta(df["rel_norm_s"], unit="s")
df["Date_dmy"] = df["Date_Time_Absolute_dmy_hmsf"]
df["Time_Absolute_hms"] = df["Date_Time_Absolute_dmy_hmsf"].dt.strftime("%H:%M:%S")
df["Time_Absolute_f"] = (df["Date_Time_Absolute_dmy_hmsf"].dt.microsecond / 1000.0).round(3)

# Rows we can actually anchor in time
df = df.dropna(subset=["Observation", "Date_Time_Absolute_dmy_hmsf"]).copy()

# Build second-level start/end from reconstructed absolute dt
df["_start_dt_sec"] = df["Date_Time_Absolute_dmy_hmsf"].dt.floor("s")
df["_end_dt_sec"] = df["_start_dt_sec"] + pd.to_timedelta(df["_dur_s_int"], unit="s")

df = df.sort_values(["Observation", "_start_dt_sec", "Date_Time_Absolute_dmy_hmsf"], kind="mergesort")



# ---------- Build sec_by_sec preserving activity + posture streams ----------
out = []
helper_cols = {"_start_dt_sec", "_dur_s", "_dur_s_int", "_end_dt_sec"}
carry_cols = [c for c in df.columns if c not in helper_cols]

activity_keys = {
    "sl- sleep", "pc- groom, health-related", "pc- other personal care",
    "ha- housework", "ha- food prep and cleanup",
    "ha- interior maintenance, repair, & decoration",
    "ha- exterior maintenance, repair, & decoration",
    "ha- lawn, garden and houseplants", "ha- animals and pets",
    "ha- household management/other household activities",
    "ca- caring for and helping children", "ca- caring for and helping adults",
    "wrk- general", "wrk- screen based",
    "edu- taking class, research, homework", "edu- extracurricular",
    "org- organizational civic, volunteer, and religious activities",
    "org - volunteer work", "org- volunteer work",
    "pur- purchasing goods and services", "eat- eating and drinking, waiting",
    "les- socializing, communicating, leisure time not screen",
    "les- screen based leisure time (tv, video game, computer)",
    "ex- participating in sport, exercise or recreation",
    "ex- attending sport, recreational event, or performance",
    "trav- passenger bus or train", "trav- driver (car/truck/motorcycle)",
    "trav- biking", "trav- walking", "trav-walking", "other- non codable",
}
posture_keys = {
    "sb-sitting", "sb- lying", "la- kneeling/ squatting", "la- stretching",
    "la- stand", "la- stand and move",
    "la- stand and move with upper body movement",
    "la- stand and move with unidentifiable upper body movement",
    "wa- walk", "wa-walk with load", "wa- ascend stairs", "wa- descend stairs",
    "wa- running", "sp- bike", "sp- other sport movement",
    "sp- muscle strengthening", "sp -kick", "sp- jump", "sp- throw",
    "private/not coded",
}

for obs, g in df.groupby("Observation", sort=False):
    g = g.copy()

    start_dt = g["_start_dt_sec"].min()
    end_dt = g["_end_dt_sec"].max()
    grid = pd.date_range(start=start_dt, end=end_dt, freq="1s")

    starts = g["_start_dt_sec"].to_numpy()
    ends = g["_end_dt_sec"].to_numpy()
    tvals = grid.to_numpy()

    # Base row selection (latest-starting covered event).
    idx_any = np.searchsorted(starts, tvals, side="right") - 1
    valid_any = idx_any >= 0
    valid_any &= tvals <= ends[np.maximum(idx_any, 0)]

    res = pd.DataFrame({"Observation": obs, "date_time_abs": grid})
    res["_sec"] = (res["date_time_abs"] - start_dt).dt.total_seconds().astype("int64")
    res["time_abs_hms"] = res["date_time_abs"].dt.strftime("%H:%M:%S")
    res["time_rel"] = (
        pd.to_timedelta(res["_sec"], unit="s")
        .astype(str)
        .str.replace("0 days ", "", regex=False)
        .str.zfill(8)
    )

    if valid_any.any():
        take_rows = g.iloc[idx_any[valid_any]][carry_cols].reset_index(drop=True)
        for c in take_rows.columns:
            if c in {"Observation"}:
                continue
            res.loc[valid_any, c] = take_rows[c].to_numpy()

    # Stream masks from raw Behavior text.
    beh_norm = (
        g["Behavior"].astype("string").str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
    )
    activity_mask = beh_norm.isin(activity_keys).to_numpy()
    posture_mask = beh_norm.isin(posture_keys).to_numpy()

    # Activity stream: keep latest-starting active row at each second.
    if activity_mask.any():
        act_idx = np.where(activity_mask)[0]
        act_starts = starts[act_idx]
        act_ends = ends[act_idx]
        idx_act = np.searchsorted(act_starts, tvals, side="right") - 1
        valid_act = idx_act >= 0
        valid_act &= tvals <= act_ends[np.maximum(idx_act, 0)]
        if valid_act.any():
            picked = act_idx[idx_act[valid_act]]
            res.loc[valid_act, "_behavior_activity_raw"] = g.iloc[picked]["Behavior"].to_numpy()

    # Posture stream: keep latest-starting active row at each second.
    if posture_mask.any():
        pos_idx = np.where(posture_mask)[0]
        pos_starts = starts[pos_idx]
        pos_ends = ends[pos_idx]
        idx_pos = np.searchsorted(pos_starts, tvals, side="right") - 1
        valid_pos = idx_pos >= 0
        valid_pos &= tvals <= pos_ends[np.maximum(idx_pos, 0)]
        if valid_pos.any():
            picked = pos_idx[idx_pos[valid_pos]]
            res.loc[valid_pos, "_behavior_posture_raw"] = g.iloc[picked]["Behavior"].to_numpy()

            # Keep posture-side modifiers when available so posture details are not lost.
            for mc in ["Modifier_1", "Modifier_2", "Modifier_3", "Modifier_4"]:
                if mc in g.columns:
                    res.loc[valid_pos, mc] = g.iloc[picked][mc].to_numpy()

    out.append(res)

sec_by_sec = pd.concat(out, ignore_index=True)
sec_by_sec = sec_by_sec.drop(columns=[c for c in ["_dur_s", "_dur_s_int"] if c in sec_by_sec.columns], errors="ignore")

# Drop uncovered seconds (no mapped event row attached).
before_rows = len(sec_by_sec)
if "id" in sec_by_sec.columns:
    sec_by_sec = sec_by_sec[sec_by_sec["id"].notna()].copy()
removed_rows = before_rows - len(sec_by_sec)
print("\nDropped uncovered rows (id is NA):", int(removed_rows))

print("sec_by_sec shape:", sec_by_sec.shape)
print("unique Observation:", sec_by_sec["Observation"].nunique())
sec_by_sec.head()


Rows with missing parsed absolute datetime: 0
Rows with negative rel_s after abs-derivation: 0
Mapping status:
map_status
one_to_one_unique_assignment    56
unmapped_extra_observation       2
missing_log_group                1

Duration abs diff summary (union coverage vs log):
count      56.000000
mean       79.928571
std       429.344103
min         0.000000
50%         1.000000
90%         9.000000
95%       127.000000
99%      1893.950000
max      3099.000000

Observations missing log mapping: 1
      Observation  id_num  do do_base
AM10DO1_R_FINAL_C      10 DO1     DO1


C:\Users\HELIOS-300\AppData\Local\Temp\ipykernel_31508\2744728748.py:106: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_union_coverage_seconds)



Dropped uncovered rows (id is NA): 6
sec_by_sec shape: (360857, 39)
unique Observation: 56


,Observation,date_time_abs,_sec,time_abs_hms,time_rel,Date_Time_Absolute_dmy_hmsf,Date_dmy,Time_Absolute_hms,Time_Absolute_f,Time_Relative_hmsf,...,rel_end_s,log_start_dt,log_stop_dt,dur_log_s,log_obs,map_status,rel_min,rel_norm_s,_behavior_activity_raw,_behavior_posture_raw
0,AM01DO1_J_FINAL_R,2017-10-03 18:44:45,0,18:44:45,00:00:00,2017-10-03 18:44:45,2017-10-03 18:44:45,18:44:45,0.0,00:00:00,...,36.0,2017-10-03 18:44:45,2017-10-03 20:47:00,7335.0,DO1,one_to_one_unique_assignment,0.0,0.0,"LES- socializing, communicating, leisure time ...",LA- stand
1,AM01DO1_J_FINAL_R,2017-10-03 18:44:46,1,18:44:46,00:00:01,2017-10-03 18:44:45,2017-10-03 18:44:45,18:44:45,0.0,00:00:00,...,36.0,2017-10-03 18:44:45,2017-10-03 20:47:00,7335.0,DO1,one_to_one_unique_assignment,0.0,0.0,"LES- socializing, communicating, leisure time ...",LA- stand
2,AM01DO1_J_FINAL_R,2017-10-03 18:44:47,2,18:44:47,00:00:02,2017-10-03 18:44:45,2017-10-03 18:44:45,18:44:45,0.0,00:00:00,...,36.0,2017-10-03 18:44:45,2017-10-03 20:47:00,7335.0,DO1,one_to_one_unique_assignment,0.0,0.0,"LES- socializing, communicating, leisure time ...",LA- stand
3,AM01DO1_J_FINAL_R,2017-10-03 18:44:48,3,18:44:48,00:00:03,2017-10-03 18:44:45,2017-10-03 18:44:45,18:44:45,0.0,00:00:00,...,36.0,2017-10-03 18:44:45,2017-10-03 20:47:00,7335.0,DO1,one_to_one_unique_assignment,0.0,0.0,"LES- socializing, communicating, leisure time ...",LA- stand
4,AM01DO1_J_FINAL_R,2017-10-03 18:44:49,4,18:44:49,00:00:04,2017-10-03 18:44:45,2017-10-03 18:44:45,18:44:45,0.0,00:00:00,...,36.0,2017-10-03 18:44:45,2017-10-03 20:47:00,7335.0,DO1,one_to_one_unique_assignment,0.0,0.0,"LES- socializing, communicating, leisure time ...",LA- stand


In [20]:
# ------------------------------------------------------------
# Carry forward Behavior + Modifiers within Observation
# (ffill only; if the first value is NaN, it stays NaN)
#
# NOTE: Comment is handled later with segment-aware logic so it
# does not bleed across activity/posture changes.
# ------------------------------------------------------------

cols_to_ffill = ["Behavior", "Modifier_1", "Modifier_2", "Modifier_3", "Modifier_4"]
cols_to_ffill = [c for c in cols_to_ffill if c in sec_by_sec.columns]

sec_by_sec = sec_by_sec.sort_values(["Observation", "date_time_abs"], kind="mergesort")
sec_by_sec[cols_to_ffill] = (
    sec_by_sec.groupby("Observation", sort=False)[cols_to_ffill].ffill()
)

# quick check: NaNs can still exist if the first event is NaN
check_cols = cols_to_ffill + (["Comment"] if "Comment" in sec_by_sec.columns else [])
sec_by_sec[check_cols].isna().sum()

Behavior           0
Modifier_1     11138
Modifier_2    360857
Modifier_3     17455
Modifier_4    359565
Comment       345769
dtype: int64

In [21]:
# ------------------------------------------------------------
# Keep only the required time columns and rename them
# ------------------------------------------------------------

# Columns to keep (if present)
keep_map = {
    "date_time_abs": "date_time",
    "time_abs_hms": "time",
    "time_rel": "rel_time",
    "Duration_sf": "duration",
}

# Drop other time-based columns if they exist
time_cols_to_drop = [
    "_sec",
    "Date_Time_Absolute_dmy_hmsf",
    "Date_dmy",
    "Time_Absolute_hms",
    "Time_Absolute_f",
    "Time_Relative_hmsf",
    "Time_Relative_hms",
    "Time_Relative_f",
    "Time_Relative_sf",
]

sec_by_sec = sec_by_sec.drop(columns=[c for c in time_cols_to_drop if c in sec_by_sec.columns])
sec_by_sec = sec_by_sec.rename(columns=keep_map)

# ------------------------------------------------------------
# Keep split sessions separate in do_session
# Priority: mapped log label (e.g., DO2_a/DO2_b) -> do -> do_base
# ------------------------------------------------------------

if "log_obs" in sec_by_sec.columns:
    sec_by_sec["do_session"] = sec_by_sec["log_obs"].astype("string").str.strip()

if "do_session" not in sec_by_sec.columns and "do" in sec_by_sec.columns:
    sec_by_sec["do_session"] = sec_by_sec["do"].astype("string").str.strip()

if "do_session" in sec_by_sec.columns and "do" in sec_by_sec.columns:
    fill_mask = sec_by_sec["do_session"].isna() | sec_by_sec["do_session"].eq("")
    sec_by_sec.loc[fill_mask, "do_session"] = sec_by_sec.loc[fill_mask, "do"].astype("string").str.strip()

if "do_session" in sec_by_sec.columns and "do_base" in sec_by_sec.columns:
    fill_mask = sec_by_sec["do_session"].isna() | sec_by_sec["do_session"].eq("")
    sec_by_sec.loc[fill_mask, "do_session"] = sec_by_sec.loc[fill_mask, "do_base"].astype("string").str.strip()

# ------------------------------------------------------------
# Drop QC-only absolute datetime column
# ------------------------------------------------------------

sec_by_sec = sec_by_sec.drop(columns=["_abs_dt_behav"], errors="ignore")

# ------------------------------------------------------------
# time_relative_new baseline (per Observation)
# ------------------------------------------------------------
sec_by_sec = sec_by_sec.sort_values(["Observation", "date_time"], kind="mergesort")
start_per_obs = sec_by_sec.groupby("Observation")["date_time"].transform("min")
secs_from_start = (sec_by_sec["date_time"] - start_per_obs).dt.total_seconds().astype("int64")
sec_by_sec["time_relative_new"] = (
    pd.Timestamp("1900-01-01") + pd.to_timedelta(secs_from_start, unit="s")
).dt.strftime("%H:%M:%S")

# Keep split sessions as-is for now (no _a/_b combine).
# Keep rows ordered for downstream export/QC.
sec_by_sec = sec_by_sec.sort_values(["id", "do_session", "date_time"], kind="mergesort")

# Quick QC: absolute time fields should be non-missing and non-negative-like.
if "date_time" in sec_by_sec.columns:
    print("Missing date_time rows:", int(sec_by_sec["date_time"].isna().sum()))
if "time" in sec_by_sec.columns:
    t = sec_by_sec["time"].astype("string").str.strip()
    print("Missing time rows:", int(t.isna().sum() + t.eq("").sum()))
    print("Rows with negative-like time strings:", int(t.str.startswith("-").fillna(False).sum()))

# Quick QC: duplicate keys at second-level
if all(c in sec_by_sec.columns for c in ["id", "do_session", "date_time"]):
    dup_mask = sec_by_sec.duplicated(subset=["id", "do_session", "date_time"], keep=False)
    dup_n = int(dup_mask.sum())
    print("Duplicate rows by [id, do_session, date_time]:", dup_n)
    if dup_n > 0:
        top_dup = (
            sec_by_sec.loc[dup_mask]
            .groupby(["id", "do_session"], dropna=False)
            .size()
            .sort_values(ascending=False)
            .head(10)
        )
        print("Top duplicate groups:")
        print(top_dup.to_string())

sec_by_sec.head()

Missing date_time rows: 0
Missing time rows: 0
Rows with negative-like time strings: 0
Duplicate rows by [id, do_session, date_time]: 0


,Observation,date_time,time,rel_time,duration,Event_Log,Behavior,Modifier_1,Modifier_2,Modifier_3,...,log_stop_dt,dur_log_s,log_obs,map_status,rel_min,rel_norm_s,_behavior_activity_raw,_behavior_posture_raw,do_session,time_relative_new
0,AM01DO1_J_FINAL_R,2017-10-03 18:44:45,18:44:45,00:00:00,35.8486,Event log,LA- stand,No movement,NaN,NaN,...,2017-10-03 20:47:00,7335.0,DO1,one_to_one_unique_assignment,0.0,0.0,"LES- socializing, communicating, leisure time ...",LA- stand,DO1,00:00:00
1,AM01DO1_J_FINAL_R,2017-10-03 18:44:46,18:44:46,00:00:01,35.8486,Event log,LA- stand,No movement,NaN,NaN,...,2017-10-03 20:47:00,7335.0,DO1,one_to_one_unique_assignment,0.0,0.0,"LES- socializing, communicating, leisure time ...",LA- stand,DO1,00:00:01
2,AM01DO1_J_FINAL_R,2017-10-03 18:44:47,18:44:47,00:00:02,35.8486,Event log,LA- stand,No movement,NaN,NaN,...,2017-10-03 20:47:00,7335.0,DO1,one_to_one_unique_assignment,0.0,0.0,"LES- socializing, communicating, leisure time ...",LA- stand,DO1,00:00:02
3,AM01DO1_J_FINAL_R,2017-10-03 18:44:48,18:44:48,00:00:03,35.8486,Event log,LA- stand,No movement,NaN,NaN,...,2017-10-03 20:47:00,7335.0,DO1,one_to_one_unique_assignment,0.0,0.0,"LES- socializing, communicating, leisure time ...",LA- stand,DO1,00:00:03
4,AM01DO1_J_FINAL_R,2017-10-03 18:44:49,18:44:49,00:00:04,35.8486,Event log,LA- stand,No movement,NaN,NaN,...,2017-10-03 20:47:00,7335.0,DO1,one_to_one_unique_assignment,0.0,0.0,"LES- socializing, communicating, leisure time ...",LA- stand,DO1,00:00:04


In [22]:
# ------------------------------------------------------------
# Last row per Observation (shows final rel_time per session)
# ------------------------------------------------------------

sec_by_sec_last = (
    sec_by_sec.sort_values(["Observation", "date_time"], kind="mergesort")
    .groupby("Observation", sort=False)
    .tail(1)
)

sec_by_sec_last[["Observation", "rel_time", "date_time", "time", "duration"]].head(20)

,Observation,rel_time,date_time,time,duration
7336,AM01DO1_J_FINAL_R,02:02:16,2017-10-03 20:47:01,20:47:01,13.48020
14601,AM01DO2_M_FINAL_R,02:01:04,2017-10-06 18:45:01,18:45:01,639.86900
21825,AM02DO1_J_FINAL_R,02:00:23,2017-07-24 15:17:33,15:17:33,294.88200
24952,AM02DO2_J_copyA_FINAL_R,00:52:06,2017-07-25 08:52:33,08:52:33,1.54829
29529,AM02DO2_J_copyB_FINAL_R,01:16:16,2017-07-25 10:12:29,10:12:29,10.87720
36745,AM03DO1_M_FINAL_R,02:00:14,2017-07-25 16:00:24,16:00:24,47.60780
43897,AM03DO2_J_FINAL_R,01:59:11,2017-07-27 15:00:40,15:00:40,266.39700
51165,AM04DO1_J_FINAL_R,02:01:07,2017-09-12 14:04:03,14:04:03,27.14390
58371,AM04DO2_M_FINAL_R,02:00:05,2017-09-17 16:16:03,16:16:03,18.84200
65640,AM05DO2re_R_FINAL_C,02:01:08,2017-09-19 21:40:01,21:40:01,185.40000


In [23]:
# ------------------------------------------------------------
# Split Behavior into Activity_Type and Posture (coded values)
# (Behavior stays as-is; new columns are filled per Observation)
# ------------------------------------------------------------

sec_by_sec["Activity_Type"] = pd.NA
sec_by_sec["Posture"] = pd.NA

# Use stream-specific behavior when available so concurrent activity+posture
# events at the same second are both preserved.
beh_activity_src = (
    sec_by_sec["_behavior_activity_raw"]
    if "_behavior_activity_raw" in sec_by_sec.columns
    else sec_by_sec["Behavior"]
)
beh_posture_src = (
    sec_by_sec["_behavior_posture_raw"]
    if "_behavior_posture_raw" in sec_by_sec.columns
    else sec_by_sec["Behavior"]
)

beh_activity_norm = (
    beh_activity_src.astype("string").str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
)
beh_posture_norm = (
    beh_posture_src.astype("string").str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
)

# Activity type mapping (Behavior -> activity_type code)
activity_map = {
    "sl- sleep": "sleep",
    "pc- groom, health-related": "pc_groom",
    "pc- other personal care": "pc_other",
    "ha- housework": "ha_housework",
    "ha- food prep and cleanup": "ha_food",
    "ha- interior maintenance, repair, & decoration": "ha_interior",
    "ha- exterior maintenance, repair, & decoration": "ha_exterior",
    "ha- lawn, garden and houseplants": "ha_lawn",
    "ha- animals and pets": "ha_pets",
    "ha- household management/other household activities": "ha_other",
    "ca- caring for and helping children": "care_children",
    "ca- caring for and helping adults": "care_adults",
    "wrk- general": "work_general",
    "wrk- screen based": "work_screen",
    "edu- taking class, research, homework": "edu_class",
    "edu- extracurricular": "edu_other",
    "org- organizational civic, volunteer, and religious activities": "com_church",
    "org - volunteer work": "com_volunteer",
    "org- volunteer work": "com_volunteer",
    "pur- purchasing goods and services": "com_purchase",
    "eat- eating and drinking, waiting": "ha_eat",
    "les- socializing, communicating, leisure time not screen": "les_social",
    "les- screen based leisure time (tv, video game, computer)": "les_screen",
    "ex- participating in sport, exercise or recreation": "ex_sport",
    "ex- attending sport, recreational event, or performance": "les_attend",
    "trav- passenger bus or train": "trav_pass",
    "trav- driver (car/truck/motorcycle)": "trav_drive",
    "trav- biking": "trav_bike",
    "trav- walking": "trav_walk",
    "trav-walking": "trav_walk",
    "other- non codable": "non_codable",
}

# Posture mapping (Behavior -> posture code)
posture_map = {
    "sb-sitting": "sitting",
    "sb- lying": "lying",
    "la- kneeling/ squatting": "kneel_squat",
    "la- stretching": "stretch",
    "la- stand": "stand",
    "la- stand and move": "stand_move",
    "la- stand and move with upper body movement": "stand_move",
    "la- stand and move with unidentifiable upper body movement": "stand_move",
    "wa- walk": "walk",
    "wa-walk with load": "walk_load",
    "wa- ascend stairs": "ascend",
    "wa- descend stairs": "descend",
    "wa- running": "running",
    "sp- bike": "biking",
    "sp- other sport movement": "sport_move",
    "sp- muscle strengthening": "muscle_strength",
    "sp -kick": "sport_move",
    "sp- jump": "sport_move",
    "sp- throw": "sport_move",
    "private/not coded": "not_coded",
}

activity_codes = beh_activity_norm.map(activity_map)
posture_codes = beh_posture_norm.map(posture_map)

# QC: show any normalized labels not covered by each stream mapping.
unmapped_activity = sorted(
    beh_activity_norm[activity_codes.isna() & beh_activity_norm.notna()].drop_duplicates().tolist()
)
unmapped_posture = sorted(
    beh_posture_norm[posture_codes.isna() & beh_posture_norm.notna()].drop_duplicates().tolist()
)
print("Unmapped activity labels:", len(unmapped_activity))
if unmapped_activity:
    print(unmapped_activity[:20])
print("Unmapped posture labels:", len(unmapped_posture))
if unmapped_posture:
    print(unmapped_posture[:20])

sec_by_sec.loc[activity_codes.notna(), "Activity_Type"] = activity_codes
sec_by_sec.loc[posture_codes.notna(), "Posture"] = posture_codes

# Carry forward within each Observation
sort_col = "date_time" if "date_time" in sec_by_sec.columns else "date_time_abs"
sec_by_sec = sec_by_sec.sort_values(["Observation", sort_col], kind="mergesort")
sec_by_sec[["Activity_Type", "Posture"]] = (
    sec_by_sec.groupby("Observation", sort=False)[["Activity_Type", "Posture"]].ffill()
)

# ------------------------------------------------------------
# Activity refinements:
# 1) EX sport/recreation rows use Modifier_2 as the specific activity label
#    (e.g., EX-hiking, EX-basketball).
# 2) Work rows keep Activity_Type as work_general/work_screen and store
#    specific work subtype in work_type from Modifier_4.
# ------------------------------------------------------------
if "Modifier_2" in sec_by_sec.columns:
    ex_mod = (
        sec_by_sec["Modifier_2"].astype("string").str.strip().str.lower()
        .str.replace(r"^ex[\s\-_]*", "", regex=True)
        .str.replace(r"[^a-z0-9]+", "-", regex=True)
        .str.strip("-")
    )
    ex_mask = sec_by_sec["Activity_Type"].eq("ex_sport") & ex_mod.notna() & ex_mod.ne("")
    sec_by_sec.loc[ex_mask, "Activity_Type"] = "EX-" + ex_mod[ex_mask]

sec_by_sec["work_type"] = pd.NA
if "Modifier_4" in sec_by_sec.columns:
    work_mod = (
        sec_by_sec["Modifier_4"].astype("string").str.strip().str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
        .str.replace(r"^work_", "", regex=True)
        .str.replace(r"^wrk_", "", regex=True)
    )
    work_mask = sec_by_sec["Activity_Type"].isin(["work_general", "work_screen"])
    valid_work = work_mask & work_mod.notna() & work_mod.ne("")
    sec_by_sec.loc[valid_work, "work_type"] = "work_" + work_mod[valid_work]

# ------------------------------------------------------------
# Comment fill rule:
#   Stop comments when a new activity or posture type starts.
#   We ffill Comment only within stable (Activity_Type, Posture)
#   segments inside each Observation.
# ------------------------------------------------------------
if "Comment" in sec_by_sec.columns:
    seg_change = (
        sec_by_sec["Activity_Type"].ne(sec_by_sec.groupby("Observation")["Activity_Type"].shift())
        | sec_by_sec["Posture"].ne(sec_by_sec.groupby("Observation")["Posture"].shift())
    )
    seg_id = seg_change.groupby(sec_by_sec["Observation"]).cumsum()
    sec_by_sec["Comment"] = sec_by_sec.groupby(
        ["Observation", seg_id], sort=False
    )["Comment"].ffill()

sec_by_sec[["Behavior", "Activity_Type", "Posture", "Comment"]].head()

Unmapped activity labels: 0
Unmapped posture labels: 0


,Behavior,Activity_Type,Posture,Comment
0,LA- stand,les_social,stand,NaN
1,LA- stand,les_social,stand,NaN
2,LA- stand,les_social,stand,NaN
3,LA- stand,les_social,stand,NaN
4,LA- stand,les_social,stand,NaN


In [24]:
# ------------------------------------------------------------
# Add broad_domain / waves_domain (from Activity_Type)
# Add posture_broad / posture_waves (from posture_wbm)
# ------------------------------------------------------------

# Activity_Type -> domains
activity_domain_map = {
    "sleep": ("sleep", "other"),
    "pc_groom": ("personal", "household"),
    "pc_other": ("personal", "household"),
    "ha_housework": ("household", "household"),
    "ha_food": ("household", "household"),
    "ha_interior": ("maintenance_repair", "household"),
    "ha_exterior": ("maintenance_repair", "household"),
    "ha_lawn": ("lawn_garden", "household"),
    "ha_pets": ("household", "household"),
    "ha_other": ("household", "household"),
    "care_children": ("household", "household"),
    "care_adults": ("household", "household"),
    "work_general": ("work_education", "occupation"),
    "work_screen": ("work_education", "occupation"), 
    "edu_class": ("work_education", "occupation"),
    "edu_other": ("work_education", "occupation"),
    "com_church": ("purchase_other", "other"),
    "com_volunteer": ("purchase_other", "other"),
    "com_purchase": ("purchase_other", "shopping"),
    "ha_eat": ("personal", "leisure_inactive"),
    "les_social": ("leisure", "leisure_inactive"),
    "les_screen": ("Leisure_Screen", "leisure_inactive"),
    "ex_sport": ("exercise", "active_time"),
    "les_attend": ("leisure", "leisure_inactive"),
    "trav_pass": ("Trav_car", "travel_inactive"),
    "trav_drive": ("Trav_car", "travel_inactive"),
    "trav_bike": ("active_transportation", "active_time"),
    "trav_walk": ("active_transportation", "active_time"),
    "trav_other": ("transportation", "other"),
    "non_codable": ("non_codable", "non_pa"),
}

sec_by_sec["broad_domain"] = sec_by_sec["Activity_Type"].map(lambda x: activity_domain_map.get(x, (pd.NA, pd.NA))[0])
sec_by_sec["waves_domain"] = sec_by_sec["Activity_Type"].map(lambda x: activity_domain_map.get(x, (pd.NA, pd.NA))[1])

# Any refined sport labels (EX-*) should stay in exercise / active_time.
ex_dyn_mask = sec_by_sec["Activity_Type"].astype("string").str.startswith("EX-").fillna(False)
sec_by_sec.loc[ex_dyn_mask, "broad_domain"] = "exercise"
sec_by_sec.loc[ex_dyn_mask, "waves_domain"] = "active_time"

# Rename Posture -> posture_wbm if needed
if "Posture" in sec_by_sec.columns and "posture_wbm" not in sec_by_sec.columns:
    sec_by_sec = sec_by_sec.rename(columns={"Posture": "posture_wbm"})

# posture_wbm -> posture_broad / posture_waves
posture_map = {
    "sitting": ("sedentary", "sedentary"),
    "lying": ("sedentary", "sedentary"),
    "kneel_squat": ("sedentary", "stationary"),
    "stretch": ("sport", "mixed_movement"),
    "stand": ("stand_move", "stationary"),
    "stand_move": ("stand_move", "stationary"),
    "walk": ("walk", "walking"),
    "walk_load": ("mod_walk", "walking"),
    "ascend": ("mod_walk", "mixed_movement"),
    "descend": ("mod_walk", "mixed_movement"),
    "running": ("running", "running"),
    "biking": ("biking", "cycling"),
    "sport_move": ("sport", "mixed_movement"),
    "muscle_strength": ("sport", "mixed_movement"),
    "not_coded": ("not_coded", "not_coded"),
}

sec_by_sec["posture_broad"] = sec_by_sec["posture_wbm"].map(lambda x: posture_map.get(x, (pd.NA, pd.NA))[0])
sec_by_sec["posture_waves"] = sec_by_sec["posture_wbm"].map(lambda x: posture_map.get(x, (pd.NA, pd.NA))[1])

# Sitting rule override for posture_waves
is_sitting = sec_by_sec["posture_wbm"].eq("sitting")
trav_mask = sec_by_sec["Activity_Type"].isin(["trav_drive", "trav_pass"])
sec_by_sec.loc[is_sitting & trav_mask, "posture_waves"] = "sed_drive"
sec_by_sec.loc[is_sitting & ~trav_mask, "posture_waves"] = "sedentary"

# posture_wbm -> sed.posture_do (user-specified mapping)
# sitting is split by activity type into sed_drive vs sedentary
sed_posture_map = {
    "lying": "sedentary",
    "kneel_squat": "active",
    "stretch": "active",
    "stand": "active",
    "stand_move": "active",
    "walk": "active",
    "walk_load": "active",
    "ascend": "active",
    "descend": "active",
    "running": "active",
    "biking": "active",
    "sport_move": "active",
    "muscle_strength": "active",
}
sec_by_sec["sed.posture_do"] = sec_by_sec["posture_wbm"].map(sed_posture_map)
sec_by_sec.loc[is_sitting & trav_mask, "sed.posture_do"] = "sed_drive"
sec_by_sec.loc[is_sitting & ~trav_mask, "sed.posture_do"] = "sedentary"

sec_by_sec[["Activity_Type", "broad_domain", "waves_domain", "posture_wbm", "posture_broad", "posture_waves", "sed.posture_do"]].head()

,Activity_Type,broad_domain,waves_domain,posture_wbm,posture_broad,posture_waves,sed.posture_do
0,les_social,leisure,leisure_inactive,stand,stand_move,stationary,active
1,les_social,leisure,leisure_inactive,stand,stand_move,stationary,active
2,les_social,leisure,leisure_inactive,stand,stand_move,stationary,active
3,les_social,leisure,leisure_inactive,stand,stand_move,stationary,active
4,les_social,leisure,leisure_inactive,stand,stand_move,stationary,active


In [25]:
# ------------------------------------------------------------
# Final clean dataframe for WAVES
# ------------------------------------------------------------

cols_order = [
    "id",
    "do_session",
    "date_time",
    "time",
    "time_relative_new",
    "Modifier_1",
    "Modifier_2",
    "Modifier_3",
    "Modifier_4",
    "Comment",
    "Activity_Type",
    "posture_wbm",
    "broad_domain",
    "waves_domain",
    "posture_broad",
    "posture_waves",
    "sed.posture_do",
]

waves_df_clean = sec_by_sec[[c for c in cols_order if c in sec_by_sec.columns]].copy()

# Keep id in AMXX format for final export (e.g., AM01, AM02)
if "id" in waves_df_clean.columns:
    id_num = pd.to_numeric(waves_df_clean["id"], errors="coerce").astype("Int64")
    waves_df_clean["id"] = id_num.map(lambda x: f"AM{int(x):02d}" if pd.notna(x) else pd.NA)

waves_df_clean_test = waves_df_clean.copy()
waves_df_clean.head()

,id,do_session,date_time,time,time_relative_new,Modifier_1,Modifier_2,Modifier_3,Modifier_4,Comment,Activity_Type,posture_wbm,broad_domain,waves_domain,posture_broad,posture_waves,sed.posture_do
0,AM01,DO1,2017-10-03 18:44:45,18:44:45,00:00:00,No movement,NaN,NaN,NaN,NaN,les_social,stand,leisure,leisure_inactive,stand_move,stationary,active
1,AM01,DO1,2017-10-03 18:44:46,18:44:46,00:00:01,No movement,NaN,NaN,NaN,NaN,les_social,stand,leisure,leisure_inactive,stand_move,stationary,active
2,AM01,DO1,2017-10-03 18:44:47,18:44:47,00:00:02,No movement,NaN,NaN,NaN,NaN,les_social,stand,leisure,leisure_inactive,stand_move,stationary,active
3,AM01,DO1,2017-10-03 18:44:48,18:44:48,00:00:03,No movement,NaN,NaN,NaN,NaN,les_social,stand,leisure,leisure_inactive,stand_move,stationary,active
4,AM01,DO1,2017-10-03 18:44:49,18:44:49,00:00:04,No movement,NaN,NaN,NaN,NaN,les_social,stand,leisure,leisure_inactive,stand_move,stationary,active


In [26]:
waves_df_clean_test.columns

Index(['id', 'do_session', 'date_time', 'time', 'time_relative_new',
       'Modifier_1', 'Modifier_2', 'Modifier_3', 'Modifier_4', 'Comment',
       'Activity_Type', 'posture_wbm', 'broad_domain', 'waves_domain',
       'posture_broad', 'posture_waves', 'sed.posture_do'],
      dtype='object')

In [27]:
# Standardize to requested output schema
waves_df_clean.rename(columns={
    "do_session": "obs",
    "time_relative_new": "rel_time",
    "Activity_Type": "activity_type",
    "waves_domain": "broad.behavior_do",
    "posture_waves": "broad.posture_do",
    "waves_sedentary": "sed.posture_do",
}, inplace=True)

# AM02 session fix requested:
# - drop AM02 DO2_b
# - remap AM02 DO2_a -> DO2
if all(c in waves_df_clean.columns for c in ["id", "obs"]):
    _id = waves_df_clean["id"].astype("string").str.strip().str.upper()
    _obs = waves_df_clean["obs"].astype("string").str.strip()
    _drop = _id.eq("AM02") & _obs.eq("DO2_b")
    print("Rows dropped for AM02 DO2_b:", int(_drop.sum()))
    waves_df_clean = waves_df_clean.loc[~_drop].copy()

    _id2 = waves_df_clean["id"].astype("string").str.strip().str.upper()
    _obs2 = waves_df_clean["obs"].astype("string").str.strip()
    _rename = _id2.eq("AM02") & _obs2.eq("DO2_a")
    waves_df_clean.loc[_rename, "obs"] = "DO2"

# Hard cutoff for AM10 DO2 at 2:01:00 duration
if all(c in waves_df_clean.columns for c in ["id", "obs", "rel_time"]):
    _id = waves_df_clean["id"].astype("string").str.strip().str.upper()
    _obs = waves_df_clean["obs"].astype("string").str.strip()
    _rel = pd.to_timedelta(waves_df_clean["rel_time"].astype(str).str.strip(), errors="coerce")
    _cut = pd.to_timedelta("02:01:00")
    _drop_after_cut = _id.eq("AM10") & _obs.eq("DO2") & _rel.gt(_cut)
    print("Rows dropped past AM10 DO2 02:01:00 cutoff:", int(_drop_after_cut.sum()))
    waves_df_clean = waves_df_clean.loc[~_drop_after_cut].copy()

# Build date from date_time (YYYY-MM-DD)
if "date_time" in waves_df_clean.columns:
    _dt = pd.to_datetime(waves_df_clean["date_time"], errors="coerce")
    waves_df_clean["date"] = _dt.dt.strftime("%Y-%m-%d")

# work_type: keep work label in activity_type, subtype from Modifier_4
waves_df_clean["work_type"] = pd.NA
if all(c in waves_df_clean.columns for c in ["activity_type", "Modifier_4"]):
    wmod = (
        waves_df_clean["Modifier_4"].astype("string").str.strip().str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
        .str.replace(r"^work_", "", regex=True)
        .str.replace(r"^wrk_", "", regex=True)
    )
    wmask = waves_df_clean["activity_type"].isin(["work_general", "work_screen"])
    valid = wmask & wmod.notna() & wmod.ne("")
    waves_df_clean.loc[valid, "work_type"] = "work_" + wmod[valid]

# intensity_do: derive from Modifier_3, then override/fill by posture rules
waves_df_clean["intensity_do"] = pd.NA
if "Modifier_3" in waves_df_clean.columns:
    mod_text = waves_df_clean["Modifier_3"].astype("string").fillna("").str.lower()
    waves_df_clean.loc[mod_text.str.contains("vigorous", na=False), "intensity_do"] = "vigorous"
    waves_df_clean.loc[
        mod_text.str.contains("moderate", na=False) & waves_df_clean["intensity_do"].isna(),
        "intensity_do",
    ] = "moderate"

if "posture_wbm" in waves_df_clean.columns:
    p = waves_df_clean["posture_wbm"].astype("string").str.strip().str.lower()

    sed_mask = p.isin(["sitting", "lying", "kneel_squat"])
    light_mask = p.isin(["stand", "stretch"])

    # Sedentary/light posture rules always override prior intensity labels.
    waves_df_clean.loc[sed_mask, "intensity_do"] = "sedentary"
    waves_df_clean.loc[light_mask, "intensity_do"] = "light"

    # Fill any remaining missing intensity from posture class.
    missing_int = waves_df_clean["intensity_do"].isna()
    waves_df_clean.loc[missing_int & p.isin(["stand_move"]), "intensity_do"] = "light"
    waves_df_clean.loc[missing_int & p.isin(["walk", "walk_load", "ascend", "descend"]), "intensity_do"] = "moderate"
    waves_df_clean.loc[
        missing_int & p.isin(["running", "biking", "sport_move", "muscle_strength"]),
        "intensity_do",
    ] = "vigorous"

    # not_coded posture has no valid intensity — clear any forward-filled value
    waves_df_clean.loc[p.eq("not_coded") | waves_df_clean["posture_wbm"].isna(), "intensity_do"] = pd.NA

# Ensure all requested columns exist
target_cols = [
    "id",
    "obs",
    "date",
    "date_time",
    "rel_time",
    "activity_type",
    "broad_domain",
    "broad.behavior_do",
    "posture_wbm",
    "posture_broad",
    "broad.posture_do",
    "sed.posture_do",
    "intensity_do",
]
for c in target_cols:
    if c not in waves_df_clean.columns:
        waves_df_clean[c] = pd.NA

# Show columns that are not part of requested final schema
leftover_cols = [c for c in waves_df_clean.columns if c not in target_cols]
print("Leftover columns dropped:", leftover_cols)

# Keep exact order requested
waves_df_clean = waves_df_clean[target_cols].copy()

Rows dropped for AM02 DO2_b: 4577
Rows dropped past AM10 DO2 02:01:00 cutoff: 89
Leftover columns dropped: ['time', 'Modifier_1', 'Modifier_2', 'Modifier_3', 'Modifier_4', 'Comment', 'work_type']


In [28]:
waves_df_clean_test2 = waves_df_clean.copy()
waves_df_clean_test2["intensity_do"].unique()

array(['light', 'moderate', 'sedentary', <NA>, 'vigorous'], dtype=object)

In [29]:
# Export cleaned AM data for comparison
output_path = "C:/Users/HELIOS-300/Desktop/WAVES/AM Full Code/Cameron_AM_Clean.csv"
waves_df_clean.fillna("NA").to_csv(output_path, index=False)
print(f"Saved: {output_path}")

# Additional export: codebook-style columns
am_codebook_df = waves_df_clean.copy()

# Core identifiers
am_codebook_df["site"] = "CP"
am_codebook_df["pid"] = am_codebook_df["id"]
am_codebook_df["observation"] = am_codebook_df["obs"].astype("string")

# Time columns
_dt = pd.to_datetime(am_codebook_df["date_time"], errors="coerce")
am_codebook_df["date"] = _dt.dt.strftime("%Y-%m-%d")
am_codebook_df["time"] = _dt.dt.strftime("%H:%M:%S")

# domain_do (codebook levels)
domain_map = {
    "leisure": "leisure",
    "Leisure_Screen": "leisure",
    "exercise": "leisure",
    "household": "household",
    "maintenance_repair": "household",
    "lawn_garden": "household",
    "personal": "household",
    "Trav_car": "transportation",
    "active_transportation": "transportation",
    "transportation": "transportation",
    "work_education": "occupation",
    "purchase_other": "other",
    "sleep": "other",
    "non_codable": "other",
}
am_codebook_df["domain_do"] = am_codebook_df["broad_domain"].map(domain_map)

# posture_do in exact codebook style
posture_do_map = {
    "sedentary": "sedentary",
    "sed_drive": "sedentary",
    "stationary": "mixed_movement",
    "mixed_movement": "mixed_movement",
    "walking": "walking",
    "running": "running",
    "cycling": "biking",
}
am_codebook_df["posture_do"] = am_codebook_df["broad.posture_do"].map(posture_do_map)

# Sedtype_do
activity_norm = am_codebook_df["activity_type"].astype("string").str.strip().str.lower()
posture_norm = am_codebook_df["posture_wbm"].astype("string").str.strip().str.lower()
vehicle_mask = activity_norm.isin(["trav_drive", "trav_pass"])
lying_mask = posture_norm.eq("lying")
sitting_mask = posture_norm.eq("sitting")

am_codebook_df["Sedtype_do"] = "non_sedentary"
am_codebook_df.loc[sitting_mask, "Sedtype_do"] = "sit_lie"
am_codebook_df.loc[lying_mask, "Sedtype_do"] = "Lying"
am_codebook_df.loc[vehicle_mask, "Sedtype_do"] = "Vehicle"

# When posture is not_coded or NA, blank all posture-dependent columns
_bad_posture = posture_norm.eq("not_coded") | am_codebook_df["posture_wbm"].isna()
am_codebook_df.loc[_bad_posture, "posture_do"]    = pd.NA
am_codebook_df.loc[_bad_posture, "Sedtype_do"]    = pd.NA
am_codebook_df.loc[_bad_posture, "intensity3_do"] = pd.NA
am_codebook_df.loc[_bad_posture, "intensity4_do"] = pd.NA
am_codebook_df.loc[_bad_posture, "steps_do"]      = pd.NA

# intensity mappings from intensity_do
# intensity4_do: 1-to-1 copy
am_codebook_df["intensity4_do"] = am_codebook_df["intensity_do"]

# intensity3_do: same as intensity_do, but combine moderate/vigorous -> mvpa
_int3 = am_codebook_df["intensity_do"].astype("string").str.strip().str.lower()
am_codebook_df["intensity3_do"] = _int3.where(~_int3.isin(["moderate", "vigorous"]), "mvpa")

# No step data for AM pipeline
am_codebook_df["steps_do"] = pd.NA
# Write missing steps as literal "NA" in AM WavesReady output.
am_codebook_df["steps_do"] = am_codebook_df["steps_do"].where(am_codebook_df["steps_do"].notna(), "NA")

codebook_cols = [
    "site",
    "pid",
    "observation",
    "date_time",
    "date",
    "time",
    "domain_do",
    "posture_do",
    "intensity3_do",
    "intensity4_do",
    "steps_do",
    "Sedtype_do",
]
am_codebook_df = am_codebook_df[codebook_cols].copy()

codebook_output_path = "C:/Users/HELIOS-300/Desktop/WAVES/AM Full Code/Cameron_AM_Clean_WavesReady.csv"
am_codebook_df.fillna("NA").to_csv(codebook_output_path, index=False)
print(f"Saved: {codebook_output_path}")

Saved: C:/Users/HELIOS-300/Desktop/WAVES/AM Full Code/Cameron_AM_Clean.csv
Saved: C:/Users/HELIOS-300/Desktop/WAVES/AM Full Code/Cameron_AM_Clean_WavesReady.csv


In [30]:
print("test complete")

test complete
